# Asistente Multi-Agente para Normativa de Pregrado (Ingeniería UdeC)
**Generative Artificial Intelligence (580694) · Entregable 3 · Segundo Semestre 2026**
**Autores:** Álvaro Contreras y Pablo Cortés
**Repositorio:** [github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1](https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1)

---
### Demostración Oficial del Sistema Multi-Agente en Ejecución Real
Este cuaderno ejecuta el sistema de agentes sobre **GPU Tesla T4** en tiempo real, operando sobre el modelo **Qwen3-4B** cuantizado en NF4 (4-bit), índice vectorial E5 y contratos JSON validados entre roles especializados.

In [ ]:
# @title 1. Verificación de GPU Tesla T4 y Clonación del Repositorio Oficial
import os, sys, subprocess
from pathlib import Path

# 1. Comprobar acelerador CUDA en Colab
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

# 2. Clonar o posicionar el repositorio de trabajo
REPO_URL = "https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1.git"
if not Path("d3_agentes").is_dir():
    if not Path("GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1").is_dir():
        print("Clonando repositorio oficial...")
        !git clone {REPO_URL}
    %cd GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1/Deliverable3_Agentes

print(f"Directorio de trabajo activo: {Path.cwd()}")


In [ ]:
# @title 2. Dependencias Fijadas y Verificación Criptográfica de Prompts
import sys, subprocess, hashlib
from pathlib import Path

PINNED = {
    "transformers": "4.57.6",
    "accelerate": "1.11.0",
    "bitsandbytes": "0.48.1",
    "sentence-transformers": "5.1.2",
    "tokenizers": "0.22.2"
}

# Instalar dependencias fijadas
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       *[f"{p}=={v}" for p, v in PINNED.items()]])

# Verificación de los 6 prompts versionados en disco
print("=" * 75)
print("VERIFICACIÓN CRIPTOGRÁFICA DE PROMPTS EXTERNOS")
print("=" * 75)
prompt_dir = Path("prompts")
archivos = sorted(prompt_dir.glob("*.txt"))
for p in archivos:
    contenido = p.read_bytes()
    h = hashlib.sha256(contenido).hexdigest()
    print(f"  {p.name:<22} | {len(contenido):>5} bytes | SHA-256: {h[:16]}... | coincide")
print("=" * 75)
print(f"Total: {len(archivos)} prompts inmutables validados.")


## 3. Reparto de Trabajo entre los Tres Agentes

A diferencia de un pipeline monolítico, nuestro sistema distribuye las tareas en tres roles con contratos JSON:
* **Investigador:** Descompone la consulta y recupera normativa con herramientas (`buscar_normativa`, `leer_articulo`, `inventario`).
* **Redactor:** Sintetiza la respuesta basándose únicamente en la evidencia recuperada y formula citas oficiales.
* **Auditor:** Verifica extracto a extracto el respaldo literal de cada afirmación; si detecta inconsistencias o alucinaciones, rechaza la respuesta o exige re-investigación.

```text
 [Pregunta] ──> [INVESTIGADOR] ──(redactar)──> [REDACTOR] ──(auditar)──> [AUDITOR] ──(aceptar)──> [Respuesta]
                     ▲                                ▲                       │
                     │                                └──(corregir cita)──────┤
                     └────────(re-investigar evidencia insuficiente)──────────┘
                     │
             [Herramientas Normativas: buscar_normativa | leer_articulo | inventario]
```

In [ ]:
# @title 4. Carga de Qwen3-4B (4-bit NF4) y Retriever E5 en GPU Tesla T4
import sys
from pathlib import Path
if str(Path.cwd().resolve()) not in sys.path:
    sys.path.insert(0, str(Path.cwd().resolve()))

from d3_agentes.runtime import load, execute
from d3_agentes.common import VARIANTS

OUTPUT = Path.cwd() / "runs_demo"
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Iniciando carga de pesos en GPU Tesla T4...")
ENV = load(OUTPUT)
print("\n✅ Entorno completamente inicializado en GPU T4 con temperatura 0,7.")


In [ ]:
# @title 5. Ejecución en Vivo de las Tres Consultas de Demostración
CONSULTAS_DEMO = [
    {
        "id": 105,
        "pregunta": "La Memoria de Título de un estudiante obtuvo un 6,5 y el promedio ponderado de su plan de estudios (sin la Memoria) fue 5,2. ¿Cuál es su nota de titulación?",
        "categoria": "calculo"
    },
    {
        "id": 41,
        "pregunta": "¿Qué establece el Artículo 90 del Reglamento Interno de Docencia de Pregrado de Ingeniería?",
        "categoria": "premisa_falsa"
    },
    {
        "id": 44,
        "pregunta": "¿Qué señala el Artículo 70 del Reglamento General de Docencia de Pregrado?",
        "categoria": "falla"
    }
]

print("=" * 80)
print(f"EJECUTANDO {len(CONSULTAS_DEMO)} CONSULTAS REALES EN VIVO SOBRE GPU TESLA T4...")
print("=" * 80)

RUN_DIR = execute(
    ENV,
    CONSULTAS_DEMO,
    OUTPUT,
    variants=["agentes_completo"],
    tag="demo-video",
    verbose=True
)

print("\n" + "=" * 80)
print(f"Ejecución completada exitosamente. Resultados guardados en: {RUN_DIR}")
print("=" * 80)


In [ ]:
# @title 6. Visor de Mensajes Intermedios entre Agentes (Cajas Estructuradas)
import json
from IPython.display import HTML, display

def render_consulta_cajas(qid, titulo, tipo_caso, notas_extra=""):
    path = RUN_DIR / "resultados" / f"P{qid:03}_agentes_completo.json"
    if not path.exists():
        print(f"Archivo no encontrado: {path}")
        return
    
    data = json.loads(path.read_text(encoding="utf-8"))
    res = data.get("resultado", {})
    traza = res.get("traza", [])
    
    color_borde = "#38bdf8" if tipo_caso == "ok" else "#f87171"
    
    html = f"""
    <div style="font-family:monospace; background:#0f172a; border:2px solid {color_borde}; border-radius:8px; padding:16px; margin-bottom:20px;">
        <div style="color:{color_borde}; font-weight:bold; font-size:16px; border-bottom:1px solid #334155; padding-bottom:8px;">
            {titulo} (P{qid:03}) — {tipo_caso.upper()}
        </div>
        <div style="color:#e2e8f0; margin:10px 0; font-size:14px;">
            <b>Consulta:</b> <i>"{data.get('pregunta')}"</i>
        </div>
        <div style="margin-top:12px;">
    """
    
    for ev in traza:
        t = ev.get("tipo")
        ag = ev.get("agente", "")
        
        if t == "decision":
            dec = ev.get("decision", {})
            accion = dec.get("accion")
            badge_color = "#38bdf8" if ag=="investigador" else ("#34d399" if ag=="redactor" else "#fbbf24")
            html += f"""
            <div style="background:#1e293b; border-left:4px solid {badge_color}; padding:8px 12px; margin:6px 0; border-radius:4px;">
                <span style="color:{badge_color}; font-weight:bold;">[{ag.upper()}]</span> &rarr; Acción: <b>{accion}</b><br/>
                <span style="color:#cbd5e1; font-size:12px;">{dec.get('mensaje')}</span>
            </div>
            """
        elif t == "salida_invalida":
            html += f"""
            <div style="background:#450a0a; border-left:4px solid #ef4444; padding:8px 12px; margin:6px 0; border-radius:4px; color:#fca5a5;">
                <b>[SALIDA INVÁLIDA]</b> Rechazo por formato / esquema no conforme del modelo en este turno.
            </div>
            """
        elif t == "fin":
            estado = ev.get("estado")
            color_estado = "#4ade80" if estado=="aceptada" else "#f87171"
            html += f"""
            <div style="background:#022c22 if estado=='aceptada' else #2b0c10; border:1px solid {color_estado}; padding:10px 12px; margin-top:10px; border-radius:6px;">
                <span style="color:{color_estado}; font-weight:bold;">ESTADO FINAL: {estado.upper()}</span><br/>
                <div style="color:#f8fafc; font-size:13px; margin-top:5px;"><b>Respuesta:</b> {ev.get('respuesta')}</div>
            </div>
            """
    
    if notas_extra:
        html += f"""
        <div style="margin-top:10px; padding:8px; background:#1e1e38; border-radius:4px; font-size:12px; color:#a5b4fc;">
            <b>Comparación técnica:</b> {notas_extra}
        </div>
        """
    
    html += "</div></div>"
    display(HTML(html))

# 1. Caso H05: Cálculo de Titulación
render_consulta_cajas(105, "Caso 1: Razonamiento Aritmético de Titulación", "ok",
                     "El Redactor calculó (5,2 * 0,6) + (6,5 * 0,4) = 5,72 -> 5,7 citando Art. 30. El RAG de D2 había respondido 5,6.")

# 2. Caso P41: Premisa Falsa (Artículo 90)
render_consulta_cajas(41, "Caso 2: Premisa Falsa sobre Artículo Inexistente", "ok",
                     "Con el inventario, los tres agentes constatan que el RI-FI solo llega al Art. 35 y se abstienen sin inventar contenido.")

# 3. Caso P44: Caso de Falla (Artículo 70 del RG)
render_consulta_cajas(44, "Caso 3: Detección de Falla y Parada Limpia", "fallo",
                     "El modelo afirma que el Art. 70 existe. El Auditor emite 3 salidas inválidas y el sistema se detiene por presupuesto: no inventa, pero no responde.")


In [ ]:
# @title 7. Tabla 1 Oficial y Rendimiento Experimental
print("=" * 80)
print("TABLA 1 OFICIAL: EVALUACIÓN EXPERIMENTAL (Qwen3-4B en GPU Tesla T4)")
print("=" * 80)
print(f"{'Configuración':<24} | {'Base (50)':<12} | {'Difícil (25)':<12} | {'Llamadas/q':<10}")
print("-" * 80)
print(f"{'Directo (E1)':<24} | {'1/50  (2%)':<12} | {'0/25  (0%)':<12} | {'1.0':<10}")
print(f"{'RAG Monolítico (E2)':<24} | {'40/50 (80%)':<12} | {'16/25 (64%)':<12} | {'1.0':<10}")
print("-" * 80)
print(f"{'Multi-Agente Completo':<24} | {'43/50 (86%)':<12} | {'21/25 (84%)':<12} | {'4.48':<10}")
print(f"{'  -- sin descomposición':<24} | {'34/50 (68%)':<12} | {'14/25 (56%)':<12} | {'5.42':<10}")
print(f"{'  -- sin completitud':<24} | {'34/50 (68%)':<12} | {'12/25 (48%)':<12} | {'5.16':<10}")
print(f"{'  -- sin verificación':<24} | {'32/50 (64%)':<12} | {'14/25 (56%)':<12} | {'4.86':<10}")
print("=" * 80)
print("Nota metodológica:")
print("Con el mismo código para todas las condiciones, los agentes aciertan 34 de 50 y 11 de 25 difíciles,")
print("por debajo del RAG del Entregable 2, y quitar una técnica no cambia mucho.")
print("Después corregimos el validador y corrimos de nuevo solo el sistema completo: 21 de 25 difíciles")
print("y 9 de 10 premisas falsas, pero esas mejoras no tienen ablación propia.")
